# Research Notebook for PISA question and answers on different languages

## Libraries

In [103]:
import os, getpass

In [104]:
from openai import OpenAI

In [105]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [106]:
from tqdm import tqdm

## Configuration

In [107]:
BASE_URL = "https://ri-delta.ai/"  # <- replace with your portal URL root
MODEL_GPT = "gpt-5"                    # <- replace with exact model id if different
MODEL_CLAUDE = "claude-sonnet-4"
MODEL_GEMINI = "gemini-2.5-pro"

In [108]:
os.environ["LLM_API_KEY"] = "sk-8b736f91707c4d1cb2042dd74c0c647b"
client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=BASE_URL + "/api"    # e.g., https://llm.company.com/v1
)

In [109]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ"
WORKSHEET_NAME = "dataset"  # change if needed
RESULTS_CSV = "llm_eval_results.csv"
SAMPLE_PER_LANGUAGE = 2     # 1 per language
MAX_LANGUAGES = 11          # 10 languages total
# LLM_TEMPERATURE = 0.2
# LLM_MAX_TOKENS = 256
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [110]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   qid          105 non-null    object
 1   language     105 non-null    object
 2   question     105 non-null    object
 3   context      105 non-null    object
 4   options      105 non-null    object
 5   gold         105 non-null    object
 6   answer_type  105 non-null    object
 7   category     105 non-null    object
 8   difficulty   105 non-null    object
 9   rationale    44 non-null     object
 10  source       105 non-null    object
dtypes: object(11)
memory usage: 9.2+ KB


## Normalize & sample

In [112]:
df["language"] = df["language"].astype(str).str.strip()
lang_groups = []
for lang, sub in df.groupby("language", sort=True):
    lang_groups.append(sub.iloc[:SAMPLE_PER_LANGUAGE])

In [113]:
sampled = pd.concat(lang_groups, ignore_index=True).iloc[:MAX_LANGUAGES]
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [114]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 11 rows across 6 languages.


,qid,language,question,gold
0,q0006,Albanian,Çfarë përmendin shkencëtarët në artikull për t...,B
1,q0007,Albanian,Çfarë provash paraqesin Carl Lipo dhe Terry Hu...,D
2,q0001,Arabic,إذا قررتْ دانا شراء السيارة (د) وباعتها بعد ثل...,C
3,q0002,Arabic,في حالة استمرار المبيعات بهذا الشكل، في أي عام...,C
4,q0001,Chinese,如果譚雅決定購買汽車 D 並於三年後在保持良好狀態下轉售，那麼這輛汽車的大約轉售價格將是多少...,C
5,q0002,Chinese,如果這銷售趨勢持續下去的話，根據該模型，哪一年DVD 銷量會少於 1 百萬張？,C
6,q0001,Czech,Jaká bude přibližná prodejní cena auta (v zede...,C
7,q0002,Czech,"Pokud vývoj prodeje bude pokračovat, v kterém ...",C
8,q0001,English,If Tania decides to buy car D and resell it af...,C
9,q0002,English,"If this sales trend continues, what will be th...",C


## Parse options

In [115]:
def parse_options(raw: str) -> dict:
    """
    Parse options when stored as a JSON list of labeled strings, e.g.:
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

    Returns a dict like:
        {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    try:
        # Try to load as JSON list
        items = json.loads(raw)
        if not isinstance(items, list):
            raise ValueError("Expected a list of options")
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    options = {}
    for item in items:
        if not isinstance(item, str):
            raise ValueError(f"Option is not a string: {item}")
        # Match patterns like "A) text", "B. text", or "C: text"
        match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
        if match:
            label, text = match.groups()
            options[label.upper()] = text.strip()
        else:
            # Fallback: assign next available letter automatically
            next_label = chr(ord('A') + len(options))
            options[next_label] = item.strip()

    return options


In [116]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [117]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [118]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds a language-agnostic but context-aware prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])
    # You can adapt language instruction if you want the rationale in the same language:
    lang = str(row["language"]).strip()

    return (
        # f"You are answering a multiple-choice question. "
        # f"Return ONLY the chosen option letter (A, B, C, ...). "
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n"
        # f"\nReply format STRICTLY:\n" 
        # f"<LETTER>\n" 
    )

In [119]:
answer_letter_regex = re.compile(r"<\s*([A-Z])\s*[.)]?\s*>")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [120]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    # temperature: float = 0.2,
    # max_tokens: int = 256,
    system_prompt: str = "You are a helpful assistant that answers multiple-choice questions. Reply format: <LETTER>.",
    retries: int = 2,
    backoff_seconds: float = 1.5,
    **kwargs,
) -> str:
    """
    Call an LLM via chat.completions and return text.
    - `prompt`: user content (string)
    - `model`: overrides global MODEL_NAME if provided
    - `temperature`, `max_tokens`: usual decoding controls
    - `system_prompt`: system role content
    - `retries`: retry on transient errors
    - `backoff_seconds`: base backoff between retries
    - `**kwargs`: forwarded to client.chat.completions.create (e.g., stop, seed)
    """
    mdl = model or MODEL_CLAUDE
    last_err = None

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                stream=False
            )

            # Add optional params only if explicitly set
            if "temperature" in kwargs and kwargs["temperature"] is not None:
                call_args["temperature"] = kwargs["temperature"]
            if "max_tokens" in kwargs and kwargs["max_tokens"] is not None:
                call_args["max_tokens"] = kwargs["max_tokens"]

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.chat.completions.create(**call_args)
            if response is None:
                return "No response from model."
            elif not hasattr(response, "choices") or not response.choices:
                return "Response object missing 'choices'."
            else:
                return(response.choices[0].message.content or "").strip()

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [121]:
def eval_rows(
        rows: pd.DataFrame, 
        cycle: int,
        model_tag: str,
        model_name: str, 
        results_path: str = RESULTS_CSV,
        sleep_s: float = 0.0, 
        retries: int = 2
    ) -> pd.DataFrame:
    results = []
    file_exists = os.path.exists(results_path)
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_tag} / {model_name}, cycle {cycle}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            row_result = {
                "qid": qid,
                "language": lang,
                "pred": "",
                "gold": gold,
                "is_correct": False,
                "error": f"OptionsParseError: {e}",
                "raw": "",
                "question": row["question"],
                "options_json": json.dumps(opts if 'opts' in locals() else {}, ensure_ascii=False),
                "model_tag": model_tag,
                "model_name": model_name,
                "cycle": cycle,
            }
            results.append(row_result)

            # Save immediately
            pd.DataFrame([row_result]).to_csv(
                results_path,
                mode="a",
                header=not file_exists,
                index=False,
                encoding="utf-8"
            )
            file_exists = True
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        err = ""
        for attempt in range(retries + 1):
            try:
                raw = llm_completion(prompt, model = model_name)
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw = ""
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        row_result = {
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
            "model_tag": model_tag,
            "model_name": model_name,
            "cycle": cycle,
        }

        results.append(row_result)

        # *** Save this row immediately ***
        pd.DataFrame([row_result]).to_csv(
            results_path,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8"
        )
        file_exists = True

        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution and results

In [122]:
MODELS_TO_TEST = [
    ("GPT", MODEL_GPT),        # (tag, model_name_for_API)
    ("Claude", MODEL_CLAUDE),
    # ("Gemini", MODEL_GEMINI)
]

In [123]:
filtered_df = df[df["language"] == "Georgian"]

In [124]:
N_CYCLES = 1  # repeat the same question to the same LLM

In [125]:
if os.path.exists(RESULTS_CSV):
    existing_results = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results from {RESULTS_CSV}: {len(existing_results)} rows")
else:
    existing_results = pd.DataFrame()
    print("No existing results file found. Starting fresh.")

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(1, N_CYCLES + 1):
        # Determine which qids are already done for this (tag, model_name, cycle)
        if existing_results.empty:
            # Nothing done yet at all
            rows_to_eval = sampled.copy()
        else:
            # Subset only rows already done for this (tag, model_name, cycle)
            subset = existing_results[
                (existing_results["model_tag"] == tag) &
                (existing_results["model_name"] == model_name) &
                (existing_results["cycle"] == cycle)
            ]

            if subset.empty:
                # No rows done yet for this model+cycle
                rows_to_eval = sampled.copy()
            else:
                # Use (qid, language) pairs as the key — more robust than qid alone
                done_pairs = set(zip(subset["qid"], subset["language"]))

                mask = ~sampled.apply(
                    lambda r: (r["qid"], r["language"]) in done_pairs,
                    axis=1
                )
                rows_to_eval = sampled[mask]

        if rows_to_eval.empty:
            print(f"  • cycle {cycle}/{N_CYCLES}: already complete, skipping")
            continue

        print(f"  • cycle {cycle}/{N_CYCLES}: evaluating {len(rows_to_eval)} questions")
        t0 = time.time()

        df_new = eval_rows(
            rows_to_eval,
            model_tag=tag,
            model_name=model_name,
            cycle=cycle,
            results_path=RESULTS_CSV
        )

        elapsed = time.time() - t0
        print(f"    Done in {elapsed:.1f}s, newly evaluated {len(df_new)} rows.")

        # Update in-memory copy so subsequent cycles/ models can see freshly written rows
        existing_results = pd.concat([existing_results, df_new], ignore_index=True)

Loaded existing results from llm_eval_results.csv: 16 rows

Evaluating GPT -> gpt-5
  • cycle 1/1: evaluating 6 questions


Evaluating GPT / gpt-5, cycle 1: 100%|██████████| 6/6 [00:40<00:00,  6.75s/it]

    Done in 40.5s, newly evaluated 6 rows.

Evaluating Claude -> claude-sonnet-4
  • cycle 1/1: already complete, skipping


In [126]:
res_df = pd.read_csv(RESULTS_CSV)
print(f"\nTotal results loaded: {len(res_df)}")


Total results loaded: 22


In [127]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
1,GPT,gpt-5,1.000000
0,Claude,claude-sonnet-4,0.727273


In [128]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
0,Albanian,1.00
1,Arabic,1.00
5,Georgian,1.00
4,English,1.00
2,Chinese,0.75
3,Czech,0.50


In [129]:
by_model_lang = (
    res_df.groupby(["model_tag","model_name","language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["model_tag","language"])
)
print("\nAccuracy by model & language:")
display(by_model_lang)


Accuracy by model & language:


,model_tag,model_name,language,accuracy
0,Claude,claude-sonnet-4,Albanian,1.0
1,Claude,claude-sonnet-4,Arabic,1.0
2,Claude,claude-sonnet-4,Chinese,0.5
3,Claude,claude-sonnet-4,Czech,0.0
4,Claude,claude-sonnet-4,English,1.0
5,Claude,claude-sonnet-4,Georgian,1.0
6,GPT,gpt-5,Albanian,1.0
7,GPT,gpt-5,Arabic,1.0
8,GPT,gpt-5,Chinese,1.0
9,GPT,gpt-5,Czech,1.0


In [130]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","question"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)

# print("\nAccuracy by question:")
# display(by_question)

accuracy_counts_total = (
    by_question["accuracy"]
    .value_counts()
    .rename_axis("accuracy")
    .reset_index(name="count")
    .sort_values("accuracy")
)

print("\nCount of total questions by accuracy:")
display(accuracy_counts_total)

accuracy_counts = (
    by_question
    .groupby(["model_tag", "model_name", "accuracy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["model_tag", "model_name", "accuracy"])
)

print("\nCount of questions by accuracy:")
display(accuracy_counts)


Count of total questions by accuracy:


,accuracy,count
1,0.0,3
0,1.0,19



Count of questions by accuracy:


,model_tag,model_name,accuracy,count
0,Claude,claude-sonnet-4,0.0,3
1,Claude,claude-sonnet-4,1.0,8
2,GPT,gpt-5,1.0,11


In [131]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 22 items: 0.864


In [132]:
for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved GPT results to: llm_eval_results__GPT.csv
Saved Claude results to: llm_eval_results__Claude.csv


,qid,language,pred,gold,is_correct,error,raw,question,options_json,model_tag,model_name,cycle
0,q0006,Albanian,B,B,True,NaN,B.,Çfarë përmendin shkencëtarët në artikull për t...,"{""A"": ""Njerëzit u vendosën në Rapa Nui qindra ...",GPT,gpt-5,1
1,q0007,Albanian,D,D,True,NaN,<D>,Çfarë provash paraqesin Carl Lipo dhe Terry Hu...,"{""A"": ""Minjtë arritën në ishull me kanoet e nj...",GPT,gpt-5,1
2,q0001,Arabic,C,C,True,NaN,C,إذا قررتْ دانا شراء السيارة (د) وباعتها بعد ثل...,"{""A"": ""١٥٧٥"", ""B"": ""٨٩٢٥"", ""C"": ""٩٠٠٠"", ""D"": ""...",GPT,gpt-5,1
3,q0002,Arabic,C,C,True,NaN,C,في حالة استمرار المبيعات بهذا الشكل، في أي عام...,"{""A"": ""٢٠١٨"", ""B"": ""٢٠١٩"", ""C"": ""٢٠٢٠"", ""D"": ""...",GPT,gpt-5,1
4,q0001,Chinese,C,C,True,NaN,C,如果譚雅決定購買汽車 D 並於三年後在保持良好狀態下轉售，那麼這輛汽車的大約轉售價格將是多少...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5,1
